# FreshMart Smart Supermarket Sales Analytics & Data Warehouse

**Personalized assignment:** Smart Supermarket Sales Analytics System

This notebook covers the assignment requirements for data cleaning, NumPy analysis, business analysis and decision making, and also demonstrates a simple **Data Warehouse using a Snowflake Schema** with fact/dimension tables and measures.

Dataset: `freshmart_sales_dataset.csv` — 1,000 supermarket transactions for July 2026.


## 1. Business Scenario

FreshMart operates supermarket branches across Sri Lanka. Management wants to understand sales, customer purchasing patterns, branch performance and product-category performance before planning the next month's inventory.

### Main warehouse concepts
- **Fact table:** stores measurable sales events.
- **Dimension tables:** store descriptive information such as date, product, customer and branch.
- **Measures:** numerical values such as Quantity, Total Sales, Cost and Profit.
- **Snowflake Schema:** dimensions are normalized into related sub-dimensions, for example Product → Category and Branch → Region.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', None)
print('Libraries imported successfully.')


## 2. Load the Dataset

If the CSV is uploaded to Colab, the notebook uses it. Otherwise, it creates the same dataset automatically using the fixed random seed used for this assignment.

In [ ]:
import os

FILE_NAME = 'freshmart_sales_dataset.csv'

if os.path.exists(FILE_NAME):
    sales_df = pd.read_csv(FILE_NAME)
    print(f'Loaded {FILE_NAME}')
else:
    try:
        from google.colab import files
        print('Please upload freshmart_sales_dataset.csv')
        uploaded = files.upload()
        sales_df = pd.read_csv(FILE_NAME)
    except Exception:
        raise FileNotFoundError('Upload freshmart_sales_dataset.csv to Colab, then run this cell again.')

sales_df['Date'] = pd.to_datetime(sales_df['Date'], errors='coerce')
print('Rows:', len(sales_df))
display(sales_df.head(10))


## 3. Initial Data Understanding

In [ ]:
print('Shape:', sales_df.shape)
print('\nData types:')
print(sales_df.dtypes)

print('\nMissing values:')
display(sales_df.isnull().sum().to_frame('Missing_Count'))

print('\nDuplicate rows:', sales_df.duplicated().sum())


## 4. Part B — Data Cleaning

The assignment asks for:
1. Missing numerical values replaced using the **Mean**.
2. Missing numerical values replaced using **Forward Fill**.
3. Missing numerical values replaced using the **Median**.
4. Extract **Month** from Date.


In [ ]:
# Keep the original dataset unchanged for comparison.
clean_df = sales_df.copy()

# 1. Mean replacement — Unit_Price
unit_price_mean = clean_df['Unit_Price'].mean()
clean_df['Unit_Price_Mean'] = clean_df['Unit_Price'].fillna(unit_price_mean)

# 2. Forward Fill — Discount
clean_df['Discount_FFill'] = clean_df['Discount'].ffill()

# 3. Median replacement — Quantity
quantity_median = clean_df['Quantity'].median()
clean_df['Quantity_Median'] = clean_df['Quantity'].fillna(quantity_median)

# 4. Extract Month from Date
clean_df['Month'] = clean_df['Date'].dt.month
clean_df['Month_Name'] = clean_df['Date'].dt.month_name()

print('Mean Unit Price:', round(unit_price_mean, 2))
print('Median Quantity:', quantity_median)
display(clean_df[['Date','Month','Month_Name','Unit_Price','Unit_Price_Mean','Discount','Discount_FFill','Quantity','Quantity_Median']].head(12))


## 5. Create Final Clean Measures

For warehouse analysis, the cleaned numerical values are used to calculate consistent measures.

In [ ]:
clean_df['Unit_Price'] = clean_df['Unit_Price_Mean']
clean_df['Discount'] = clean_df['Discount_FFill']
clean_df['Quantity'] = clean_df['Quantity_Median']

clean_df['Total_Sales_Clean'] = clean_df['Unit_Price'] * clean_df['Quantity'] * (1 - clean_df['Discount'])
clean_df['Profit_Clean'] = clean_df['Total_Sales_Clean'] - clean_df['Cost'].fillna(clean_df['Cost'].median())

print('Remaining missing values in key numerical columns:')
display(clean_df[['Unit_Price','Quantity','Discount','Total_Sales_Clean','Profit_Clean']].isnull().sum().to_frame('Missing'))


## 6. Part C — NumPy Analysis

Required calculations:
- Maximum Unit Price
- Minimum Unit Price
- Average Unit Price
- Median Quantity
- Standard Deviation of Total Sales

In [ ]:
unit_prices = clean_df['Unit_Price'].to_numpy(dtype=float)
quantities = clean_df['Quantity'].to_numpy(dtype=float)
total_sales = clean_df['Total_Sales_Clean'].to_numpy(dtype=float)

print('Maximum Unit Price:', round(np.max(unit_prices), 2))
print('Minimum Unit Price:', round(np.min(unit_prices), 2))
print('Average Unit Price:', round(np.mean(unit_prices), 2))
print('Median Quantity:', round(np.median(quantities), 2))
print('Standard Deviation of Total Sales:', round(np.std(total_sales), 2))


## 7. Part D — Business Analysis

In [ ]:
# 1. Lowest selling product category
category_sales = clean_df.groupby('Product_Category')['Total_Sales_Clean'].sum().sort_values()
print('1. Lowest selling category:', category_sales.index[0], '| Revenue = Rs.', round(category_sales.iloc[0], 2))

# 2. Branch with highest average discount
branch_discount = clean_df.groupby('Branch')['Discount'].mean().sort_values(ascending=False)
print('2. Highest average discount branch:', branch_discount.index[0], '| Average =', round(branch_discount.iloc[0] * 100, 2), '%')

# 3. Top 15 transactions
print('\n3. Top 15 transactions:')
display(clean_df.nlargest(15, 'Total_Sales_Clean')[['Sale_ID','Date','Branch','Product','Quantity','Discount','Total_Sales_Clean']])

# 4. Highest sales day
daily_sales = clean_df.groupby('Date')['Total_Sales_Clean'].sum().sort_values(ascending=False)
print('4. Highest sales day:', daily_sales.index[0].date(), '| Revenue = Rs.', round(daily_sales.iloc[0], 2))

# 5. Highest selling product category
print('5. Highest selling category:', category_sales.index[-1], '| Revenue = Rs.', round(category_sales.iloc[-1], 2))

# 6. Revenue by branch
revenue_by_branch = clean_df.groupby('Branch')['Total_Sales_Clean'].sum().sort_values(ascending=False)
print('\n6. Revenue by branch:')
display(revenue_by_branch.to_frame('Revenue_Rs'))

# 7. Highest revenue branch
print('7. Highest revenue branch:', revenue_by_branch.index[0])

# 8. Category with highest average quantity
avg_qty_category = clean_df.groupby('Product_Category')['Quantity'].mean().sort_values(ascending=False)
print('8. Highest average quantity category:', avg_qty_category.index[0], '| Average quantity =', round(avg_qty_category.iloc[0], 2))

# 9. Lowest sales day
print('9. Lowest sales day:', daily_sales.index[-1].date(), '| Revenue = Rs.', round(daily_sales.iloc[-1], 2))

# 10. Bottom 10 transactions
print('\n10. Bottom 10 transactions:')
display(clean_df.nsmallest(10, 'Total_Sales_Clean')[['Sale_ID','Date','Branch','Product','Quantity','Discount','Total_Sales_Clean']])


## 8. Part E — Decision Making

In [ ]:
# 1. Branch deserving expansion — highest revenue with strong average profit
branch_summary = clean_df.groupby('Branch').agg(
    Revenue=('Total_Sales_Clean','sum'),
    Profit=('Profit_Clean','sum'),
    Avg_Discount=('Discount','mean')
).sort_values('Revenue', ascending=False)
print('1. Branch deserving expansion:', branch_summary.index[0])

# 2. Payment method to promote — most revenue
payment_summary = clean_df.groupby('Payment_Method')['Total_Sales_Clean'].sum().sort_values(ascending=False)
print('2. Payment method to promote:', payment_summary.index[0])

# 3. Two strategies to reduce operational costs
print('3. Cost-reduction strategies:')
print('   - Reduce excessive discounting in branches with high average discounts.')
print('   - Use branch/category demand patterns to avoid overstocking and unnecessary inventory costs.')

# 4. Identify one anomaly using IQR on Unit Price
q1 = clean_df['Unit_Price'].quantile(0.25)
q3 = clean_df['Unit_Price'].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
anomalies = clean_df[clean_df['Unit_Price'] > upper]
print('4. Number of Unit Price anomalies:', len(anomalies))
display(anomalies[['Sale_ID','Date','Branch','Product','Unit_Price','Quantity','Total_Sales_Clean']].head())

# 5. Branch that should reduce inventory — lowest revenue
print('5. Branch that should reduce inventory:', branch_summary.index[-1])


## 9. Data Warehouse Design

The raw sales data is converted into a warehouse model.

### Fact table
**fact_sales** contains the business event and measures:
- sale_id
- date_key
- product_key
- customer_key
- branch_key
- payment_key
- quantity
- unit_price
- discount
- total_sales
- cost
- profit

### Dimension tables
- **dim_date** → date information
- **dim_product** → product information and category key
- **dim_category** → product category information
- **dim_customer** → customer information
- **dim_branch** → branch information and region key
- **dim_region** → region information
- **dim_payment** → payment method information

This creates a **Snowflake Schema** because Product and Branch dimensions are normalized into Category and Region sub-dimensions.

In [ ]:
# -------------------------
# Dimension: Category
# -------------------------
category_names = sorted(clean_df['Product_Category'].dropna().unique())
dim_category = pd.DataFrame({
    'category_key': range(1, len(category_names) + 1),
    'category_name': category_names
})

# -------------------------
# Dimension: Product
# -------------------------
dim_product = clean_df[['Product_ID','Product','Product_Category']].drop_duplicates().copy()
dim_product = dim_product.merge(dim_category, left_on='Product_Category', right_on='category_name', how='left')
dim_product = dim_product.reset_index(drop=True)
dim_product['product_key'] = np.arange(1, len(dim_product) + 1)
dim_product = dim_product[['product_key','Product_ID','Product','category_key']]

# -------------------------
# Dimension: Region
# -------------------------
region_names = sorted(clean_df['Region'].dropna().unique())
dim_region = pd.DataFrame({
    'region_key': range(1, len(region_names) + 1),
    'region_name': region_names
})

# -------------------------
# Dimension: Branch
# -------------------------
dim_branch = clean_df[['Branch','Region']].drop_duplicates().copy()
dim_branch = dim_branch.merge(dim_region, left_on='Region', right_on='region_name', how='left')
dim_branch = dim_branch.reset_index(drop=True)
dim_branch['branch_key'] = np.arange(1, len(dim_branch) + 1)
dim_branch = dim_branch[['branch_key','Branch','region_key']]

# -------------------------
# Dimension: Customer
# -------------------------
customer_names = sorted(clean_df['Customer'].dropna().unique())
dim_customer = pd.DataFrame({
    'customer_key': range(1, len(customer_names) + 1),
    'customer_id': customer_names
})

# -------------------------
# Dimension: Payment
# -------------------------
payment_names = sorted(clean_df['Payment_Method'].dropna().unique())
dim_payment = pd.DataFrame({
    'payment_key': range(1, len(payment_names) + 1),
    'payment_method': payment_names
})

# -------------------------
# Dimension: Date
# -------------------------
unique_dates = pd.DataFrame({'full_date': sorted(clean_df['Date'].dropna().unique())})
dim_date = unique_dates.copy()
dim_date['date_key'] = np.arange(1, len(dim_date) + 1)
dim_date['day'] = dim_date['full_date'].dt.day
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['month_name'] = dim_date['full_date'].dt.month_name()
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['day_name'] = dim_date['full_date'].dt.day_name()
dim_date = dim_date[['date_key','full_date','day','month','month_name','year','day_name']]

print('Dimension tables created successfully.')


## 10. Create the Fact Table

The fact table contains foreign keys to the dimensions plus the measures.

In [ ]:
fact_sales = clean_df.copy()

fact_sales = fact_sales.merge(dim_date[['date_key','full_date']], left_on='Date', right_on='full_date', how='left')
fact_sales = fact_sales.merge(dim_product[['product_key','Product_ID']], on='Product_ID', how='left')
fact_sales = fact_sales.merge(dim_customer[['customer_key','customer_id']], left_on='Customer', right_on='customer_id', how='left')
fact_sales = fact_sales.merge(dim_branch[['branch_key','Branch']], on='Branch', how='left')
fact_sales = fact_sales.merge(dim_payment[['payment_key','payment_method']], left_on='Payment_Method', right_on='payment_method', how='left')

fact_sales = fact_sales[[
    'Sale_ID','date_key','product_key','customer_key','branch_key','payment_key',
    'Quantity','Unit_Price','Discount','Total_Sales_Clean','Cost','Profit_Clean'
]].rename(columns={
    'Quantity':'quantity',
    'Unit_Price':'unit_price',
    'Discount':'discount',
    'Total_Sales_Clean':'total_sales',
    'Cost':'cost',
    'Profit_Clean':'profit'
})

print('Fact rows:', len(fact_sales))
display(fact_sales.head(10))


## 11. Snowflake Schema Relationship

```text
                    dim_date
                       |
                       |
dim_product ---- fact_sales ---- dim_customer
     |
dim_category

dim_branch ---- fact_sales
     |
dim_region

dim_payment ---- fact_sales
```

**Measures:** quantity, unit_price, discount, total_sales, cost, profit.

## 12. Warehouse Measures / KPI Analysis

In [ ]:
total_revenue = fact_sales['total_sales'].sum()
total_profit = fact_sales['profit'].sum()
total_quantity = fact_sales['quantity'].sum()
total_transactions = fact_sales['Sale_ID'].nunique()
average_order_value = total_revenue / total_transactions

kpis = pd.DataFrame({
    'Measure': ['Total Revenue','Total Profit','Total Quantity','Total Transactions','Average Order Value'],
    'Value': [total_revenue,total_profit,total_quantity,total_transactions,average_order_value]
})
display(kpis)


## 13. Warehouse Queries

In [ ]:
# Revenue by branch using the warehouse tables
branch_report = fact_sales.merge(dim_branch, on='branch_key') \
    .merge(dim_region, on='region_key') \
    .groupby(['Branch','region_name'], as_index=False)['total_sales'].sum() \
    .sort_values('total_sales', ascending=False)

print('Revenue by Branch and Region')
display(branch_report)

# Revenue by category using Product -> Category snowflake relationship
category_report = fact_sales.merge(dim_product, on='product_key') \
    .merge(dim_category, on='category_key') \
    .groupby('category_name', as_index=False)['total_sales'].sum() \
    .sort_values('total_sales', ascending=False)

print('Revenue by Category')
display(category_report)


## 14. Visualization

In [ ]:
monthly_sales = clean_df.groupby('Date')['Total_Sales_Clean'].sum().sort_index()

plt.figure(figsize=(12,5))
plt.plot(monthly_sales.index, monthly_sales.values, marker='o')
plt.title('Daily Sales Trend - July 2026')
plt.xlabel('Date')
plt.ylabel('Sales (Rs.)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9,5))
plt.bar(category_report['category_name'], category_report['total_sales'])
plt.title('Revenue by Product Category')
plt.xlabel('Category')
plt.ylabel('Revenue (Rs.)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 15. Export Warehouse Tables

These CSV files can be used as evidence of the warehouse design.

In [ ]:
output_dir = 'freshmart_warehouse_output'
os.makedirs(output_dir, exist_ok=True)

tables = {
    'fact_sales': fact_sales,
    'dim_date': dim_date,
    'dim_product': dim_product,
    'dim_category': dim_category,
    'dim_customer': dim_customer,
    'dim_branch': dim_branch,
    'dim_region': dim_region,
    'dim_payment': dim_payment,
}

for name, table in tables.items():
    table.to_csv(f'{output_dir}/{name}.csv', index=False)

print('Exported warehouse tables:')
for name in tables:
    print('-', f'{output_dir}/{name}.csv')


## 16. Final Summary

The notebook demonstrates the complete flow:

**Raw Dataset → Data Cleaning → NumPy Analysis → Business Analysis → Decision Making → Data Warehouse → Fact & Dimension Tables → Snowflake Schema → Measures/KPIs → Visualization → Export**.

### Key definitions for viva
- **Fact table:** central table containing measurable business events.
- **Dimension table:** descriptive table used to filter/group facts.
- **Measure:** numeric value that can be aggregated, such as SUM(Revenue).
- **Star Schema:** fact table directly connected to denormalized dimensions.
- **Snowflake Schema:** normalized dimensions connected to sub-dimensions.
